In [5]:
import os
import pandas as pd

data_dir = "E:\\codesoft tasks\\face recognition\\lfw-deepfunneled\\lfw-deepfunneled"   # change if needed

data = []

for person_name in os.listdir(data_dir):
    person_path = os.path.join(data_dir, person_name)
    
    if os.path.isdir(person_path):
        
        # Get all valid image files first
        images = [
            img for img in os.listdir(person_path)
            if img.lower().endswith(('.png', '.jpg', '.jpeg'))
        ]
        
        num_images = len(images)  # count images
        
        # Add each image with count
        for img_name in images:
            img_path = os.path.join(person_path, img_name)
            
            data.append({
                "person": person_name,
                "image_path": img_path,
                "num_images": num_images
            })

# Create DataFrame
df = pd.DataFrame(data)

df.head()

,person,image_path,num_images
0,German_Khan,E:\codesoft tasks\face recognition\lfw-deepfun...,1
1,Stefano_Gabbana,E:\codesoft tasks\face recognition\lfw-deepfun...,1
2,Dragan_Covic,E:\codesoft tasks\face recognition\lfw-deepfun...,1
3,Jeff_Hornacek,E:\codesoft tasks\face recognition\lfw-deepfun...,1
4,Sureyya_Ayhan,E:\codesoft tasks\face recognition\lfw-deepfun...,1


In [6]:
df1=df[df['num_images']>=5]

In [7]:
df1['person'].unique().shape

(423,)

In [8]:
# ================================
# ✅ 1. Imports
# ================================
import os
import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm import tqdm
from sklearn.preprocessing import Normalizer


# ================================
# ✅ 2. Load FaceNet Model
# ================================
from facenet_pytorch import MTCNN, InceptionResnetV1

device = 'cuda' if torch.cuda.is_available() else 'cpu'

mtcnn = MTCNN(image_size=160, margin=20, device=device)

model = InceptionResnetV1(pretrained='vggface2').eval().to(device)




# ================================
# ✅ 4. Generate Embeddings (TRAINING)
# ================================
embeddings = []
labels = []

for i, row in tqdm(df1.iterrows(), total=len(df1)):
    img_path = row['image_path']
    person = row['person']
    
    try:
        img = Image.open(img_path).convert('RGB')
    except:
        continue
    
    # 🔥 Face detection + alignment
    face = mtcnn(img)
    
    if face is None:
        continue
    
    face = face.unsqueeze(0).to(device)
    
    # 🔥 FaceNet embedding
    with torch.no_grad():
        emb = model(face)
    
    embeddings.append(emb.squeeze().cpu().numpy())
    labels.append(person)

X_emb = np.array(embeddings)
y = np.array(labels)

print("Embeddings shape:", X_emb.shape)


# ================================
# ✅ 5. Normalize Embeddings
# ================================
normalizer = Normalizer(norm='l2')
X_emb = normalizer.transform(X_emb)


# ================================
# ✅ 6. Cosine Similarity
# ================================
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


# ================================
# ✅ 7. Get embedding for NEW image
# ================================
def get_embedding(img_path):
    try:
        img = Image.open(img_path).convert('RGB')
    except:
        return None
    
    face = mtcnn(img)
    
    if face is None:
        return None
    
    face = face.unsqueeze(0).to(device)
    
    with torch.no_grad():
        emb = model(face)
    
    emb = emb.squeeze().cpu().numpy()
    
    # normalize
    emb = emb / np.linalg.norm(emb)
    
    return emb


# ================================
# ✅ 8. Face Recognition
# ================================
def recognize_face(img_path, threshold=0.6):
    query_emb = get_embedding(img_path)
    
    if query_emb is None:
        return "No face detected", 0
    
    best_score = -1
    best_label = "Unknown"
    
    for emb, label in zip(X_emb, y):
        score = cosine_similarity(query_emb, emb)
        
        if score > best_score:
            best_score = score
            best_label = label
    
    if best_score >= threshold:
        return best_label, best_score
    else:
        return "Unknown", best_score



100%|██████████| 5985/5985 [17:06<00:00,  5.83it/s]    

Embeddings shape: (5985, 512)


In [13]:

# ================================
# ✅ 9. Test on Image
# ================================
test_image = "lfw-deepfunneled\\lfw-deepfunneled\\Kalpana_Chawla\\Kalpana_Chawla_0004.jpg"   # change this

name, score = recognize_face(test_image)

print("Prediction:", name)
print("Similarity:", score)



Prediction: Kalpana_Chawla
Similarity: 1.0000001


In [17]:

# ================================
# ✅ 10. Threshold Tuning
# ================================
def test_thresholds(thresholds=np.arange(0.4, 0.8, 0.05)):
    results = []
    
    for t in thresholds:
        correct = 0
        total = 0
        
        for i in range(len(X_emb)):
            for j in range(i+1, len(X_emb)):
                
                sim = cosine_similarity(X_emb[i], X_emb[j])
                
                same = (y[i] == y[j])
                pred = sim >= t
                
                if pred == same:
                    correct += 1
                total += 1
        
        acc = correct / total
        results.append((t, acc))
    
    return results


# ================================
# ✅ 11. Run Threshold Evaluation
# ================================
results = test_thresholds()

for t, acc in results:
    print(f"Threshold: {t:.2f}, Accuracy: {acc:.4f}")

Threshold: 0.40, Accuracy: 0.9787
Threshold: 0.45, Accuracy: 0.9894
Threshold: 0.50, Accuracy: 0.9948
Threshold: 0.55, Accuracy: 0.9973
Threshold: 0.60, Accuracy: 0.9983
Threshold: 0.65, Accuracy: 0.9984
Threshold: 0.70, Accuracy: 0.9978
Threshold: 0.75, Accuracy: 0.9962


In [11]:
%pip install streamlit facenet-pytorch pillow numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
# ================================
# ✅ Save Database
# ================================
def save_database(X_emb, y, path="face_database.pkl"):
    with open(path, "wb") as f:
        pickle.dump({
            "embeddings": X_emb,
            "labels": y
        }, f)



In [14]:
import pickle

# Verify normalized before saving
import numpy as np
norms = np.linalg.norm(X_emb, axis=1)
print("Norm range:", norms.min(), "to", norms.max())  # Should be ~1.0 for all

with open("face_database.pkl", "wb") as f:
    pickle.dump({"embeddings": X_emb, "labels": y}, f)

print("✅ Saved", X_emb.shape, "embeddings for", len(set(y)), "people")

Norm range: 0.9999998 to 1.0000001
✅ Saved (5985, 512) embeddings for 423 people


✅ Embeddings are L2-normalized. Proceeding to save.
✅ Database saved to 'face_database.pkl'
   Embeddings shape : (5985, 512)
   Number of labels : 5985
   Unique persons   : 423

📦 Database Verification: 'face_database.pkl'
   Embeddings shape : (5985, 512)
   Unique persons   : 423
   Norm min/max     : 1.000000 / 1.000000
   L2 Normalized    : ✅ YES


True

In [16]:
%run -i save_model.py

✅ Embeddings are L2-normalized. Proceeding to save.
✅ Database saved to 'face_database.pkl'
   Embeddings shape : (5985, 512)
   Number of labels : 5985
   Unique persons   : 423

📦 Database Verification: 'face_database.pkl'
   Embeddings shape : (5985, 512)
   Unique persons   : 423
   Norm min/max     : 1.000000 / 1.000000
   L2 Normalized    : ✅ YES
